In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:llama-3.3-70b-versatile")

# Tools

In [21]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """ Get the weather at location"""
    return f"It's sunny in {location}."

model_with_tools = model.bind_tools([get_weather])

In [11]:
res = model_with_tools.invoke("What's weather in Pune")

In [12]:
res.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Pune'},
  'id': '2aa2405qa',
  'type': 'tool_call'}]

### Tool execution Loop

In [ ]:
from langchain_core.messages import HumanMessage

# step 1: model generates tool calls
messages = [HumanMessage(content="What's current weather in Banglore")]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)


# step 3: Provide result back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response)


[HumanMessage(content="What's current weather in Banglore", additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '458gpfh03', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 217, 'total_tokens': 232, 'completion_time': 0.033813128, 'completion_tokens_details': None, 'prompt_time': 0.011372008, 'prompt_tokens_details': None, 'queue_time': 0.057252911, 'total_time': 0.045185136}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d5a4b-29d0-7eb3-a50a-69b51120a5a9-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': '458gpfh03', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 217, 'output_tokens': 15, 't